<a href="https://colab.research.google.com/github/kauazin05/CIAO_KKG_2026/blob/main/lab1_aula_10_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-fuzzy -q

In [2]:
!mkdir -p src

In [3]:
%%writefile src/model.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

def criar_sistema_fuzzy():
    impacto = ctrl.Antecedent(np.arange(0, 101, 1), 'impacto')
    explorabilidade = ctrl.Antecedent(np.arange(0, 11, 1), 'explorabilidade')
    urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

    urgencia.defuzzify_method = 'centroid'

    impacto['baixo'] = fuzzy.trimf(impacto.universe, [0, 0, 40])
    impacto['medio'] = fuzzy.trimf(impacto.universe, [20, 50, 80])
    impacto['alto'] = fuzzy.trapmf(impacto.universe, [60, 80, 100, 100])

    explorabilidade['baixa'] = fuzzy.trimf(explorabilidade.universe, [0, 0, 4])
    explorabilidade['moderada'] = fuzzy.trimf(explorabilidade.universe, [2, 5, 8])
    explorabilidade['alta'] = fuzzy.trimf(explorabilidade.universe, [6, 10, 10])

    urgencia['baixa'] = fuzzy.trimf(urgencia.universe, [0, 0, 30])
    urgencia['media'] = fuzzy.trimf(urgencia.universe, [20, 50, 70])
    urgencia['alta'] = fuzzy.trimf(urgencia.universe, [60, 80, 90])
    urgencia['critica'] = fuzzy.trapmf(urgencia.universe, [80, 90, 100, 100])

    return impacto, explorabilidade, urgencia

Writing src/model.py


In [4]:
%%writefile src/model_a.py
from pathlib import Path
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

def criar_sistema_fuzzy_a():
    impacto = ctrl.Antecedent(np.arange(0, 101, 1), 'impacto')
    explorabilidade = ctrl.Antecedent(np.arange(0, 11, 1), 'explorabilidade')
    urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

    urgencia.defuzzify_method = 'centroid'

    impacto['baixo'] = fuzzy.trimf(impacto.universe, [0, 0, 40])
    impacto['medio'] = fuzzy.trimf(impacto.universe, [20, 50, 80])
    impacto['alto'] = fuzzy.trapmf(impacto.universe, [60, 80, 100, 100])

    explorabilidade['baixa'] = fuzzy.trimf(explorabilidade.universe, [0, 0, 4])
    explorabilidade['moderada'] = fuzzy.trimf(explorabilidade.universe, [2, 5, 8])
    explorabilidade['alta'] = fuzzy.trimf(explorabilidade.universe, [6, 10, 10])

    urgencia['baixa'] = fuzzy.trapmf(urgencia.universe, [0, 0, 5, 30])
    urgencia['media'] = fuzzy.trimf(urgencia.universe, [20, 50, 70])
    urgencia['alta'] = fuzzy.trimf(urgencia.universe, [60, 80, 90])
    urgencia['critica'] = fuzzy.trapmf(urgencia.universe, [80, 95, 100, 100])

    return impacto, explorabilidade, urgencia

Writing src/model_a.py


In [5]:
%%writefile engine_a.py
from pathlib import Path
import numpy as np
from skfuzzy import control as ctrl
from src.model_a import criar_sistema_fuzzy_a


def construir_motor_a():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy_a()
    regras = [
        ctrl.Rule(impacto['baixo'] & explorabilidade['baixa'], urgencia['baixa']),
        ctrl.Rule(impacto['baixo'] & explorabilidade['moderada'], urgencia['baixa']),
        ctrl.Rule(impacto['baixo'] & explorabilidade['alta'], urgencia['media']),
        ctrl.Rule(impacto['medio'] & explorabilidade['baixa'], urgencia['media']),
        ctrl.Rule(impacto['medio'] & explorabilidade['moderada'], urgencia['alta']),
        ctrl.Rule(impacto['medio'] & explorabilidade['alta'], urgencia['alta']),
        ctrl.Rule(impacto['alto'] & explorabilidade['baixa'], urgencia['critica']),
        ctrl.Rule(impacto['alto'] & explorabilidade['moderada'], urgencia['critica']),
        ctrl.Rule(impacto['alto'] & explorabilidade['alta'], urgencia['critica']),
    ]
    sistema_controle = ctrl.ControlSystem(regras)
    return ctrl.ControlSystemSimulation(sistema_controle)


if __name__ == "__main__":
    sim = construir_motor_a()
    for nome, imp, exp in [("Mínimo Absoluto", 0.0, 0.0), ("Máximo Absoluto", 100.0, 10.0)]:
        sim.input['impacto'] = imp
        sim.input['explorabilidade'] = exp
        sim.compute()
        print(f"[ABORDAGEM A] [{nome}] -> Imp: {imp}% | Exp: {exp} => Urgência: {sim.output['urgencia']:.2f}%")

Writing engine_a.py


In [6]:
!python engine_a.py

[ABORDAGEM A] [Mínimo Absoluto] -> Imp: 0.0% | Exp: 0.0 => Urgência: 10.24%
[ABORDAGEM A] [Máximo Absoluto] -> Imp: 100.0% | Exp: 10.0 => Urgência: 93.00%


In [9]:
%%writefile engine_b.py
from pathlib import Path
import numpy as np
from skfuzzy import control as ctrl
from src.model import criar_sistema_fuzzy


def construir_motor_b():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy()
    urgencia.defuzzify_method = 'mom'  # Mean of Maximum

    regras = [
        ctrl.Rule(impacto['baixo'] & explorabilidade['baixa'], urgencia['baixa']),
        ctrl.Rule(impacto['baixo'] & explorabilidade['moderada'], urgencia['baixa']),
        ctrl.Rule(impacto['baixo'] & explorabilidade['alta'], urgencia['media']),
        ctrl.Rule(impacto['medio'] & explorabilidade['baixa'], urgencia['media']),
        ctrl.Rule(impacto['medio'] & explorabilidade['moderada'], urgencia['alta']),
        ctrl.Rule(impacto['medio'] & explorabilidade['alta'], urgencia['alta']),
        ctrl.Rule(impacto['alto'] & explorabilidade['baixa'], urgencia['critica']),
        ctrl.Rule(impacto['alto'] & explorabilidade['moderada'], urgencia['critica']),
        ctrl.Rule(impacto['alto'] & explorabilidade['alta'], urgencia['critica']),
    ]
    sistema_controle = ctrl.ControlSystem(regras)
    return ctrl.ControlSystemSimulation(sistema_controle)


if __name__ == "__main__":
    sim = construir_motor_b()
    casos = [("Minimo Absoluto", 0.0, 0.0), ("Maximo Absoluto", 100.0, 10.0), ("Medio Impacto + Moderado", 50.0, 5.0)]
    for nome, imp, exp in casos:
        sim.input['impacto'] = imp
        sim.input['explorabilidade'] = exp
        sim.compute()
        print(f"[ABORDAGEM B - MOM] [{nome}] -> Imp: {imp}% | Exp: {exp} => Urgencia: {sim.output['urgencia']:.2f}%")

Overwriting engine_b.py


In [10]:
!python engine_b.py

[ABORDAGEM B - MOM] [Minimo Absoluto] -> Imp: 0.0% | Exp: 0.0 => Urgencia: 0.00%
[ABORDAGEM B - MOM] [Maximo Absoluto] -> Imp: 100.0% | Exp: 10.0 => Urgencia: 95.00%
[ABORDAGEM B - MOM] [Medio Impacto + Moderado] -> Imp: 50.0% | Exp: 5.0 => Urgencia: 80.00%
